# Cross-Dataset Validation: Multi-Seed XAI Protocol on IoT Agriculture 2024 (Iraq)
Run on Kaggle with GPU T4 x2 accelerator.

Dataset: https://www.kaggle.com/datasets/wisam1985/iot-agriculture-2024
Source: Smart greenhouse, Tikrit University, Iraq (arid climate)
Target: Watering plant pump (ON/OFF → 0/1)
Features: temperature, humidity, water_level, N, P, K

This notebook validates our multi-seed XAI protocol on an independent dataset
from a different greenhouse, climate, and crop to demonstrate generalizability.

In [ ]:
import numpy as np
import pandas as pd
import json
import os
import warnings
warnings.filterwarnings('ignore')

## 1. LOAD AND PREPROCESS DATASET

In [ ]:
print("=" * 60)
print("1. LOADING IoT Agriculture 2024 Dataset")
print("=" * 60)

# Load dataset
df = pd.read_csv('/kaggle/input/datasets/wisam1985/iot-agriculture-2024/IoTProcessed_Data.csv')

# Strip whitespace from column names
df.columns = df.columns.str.strip()
print(f"Raw shape: {df.shape}")
print(f"Columns: {list(df.columns)}")
print(df.head())
print(df.dtypes)
print(df.describe())

# Sort by date
df['date'] = pd.to_datetime(df['date'], errors='coerce')
df = df.sort_values('date').reset_index(drop=True)

# Exact column names (note: "tempreature" is a typo in the dataset)
feature_cols = ['tempreature', 'humidity', 'water_level', 'N', 'P', 'K']
target_col = 'Watering_plant_pump_ON'

print(f"\nFeatures: {feature_cols}")
print(f"Target: {target_col}")
print(f"Target distribution:\n{df[target_col].value_counts()}")

# Drop rows with missing values
df_clean = df[feature_cols + [target_col]].dropna()
print(f"After dropping NaN: {df_clean.shape}")

# MinMax scale features to [0, 1]
from sklearn.preprocessing import MinMaxScaler

scaler = MinMaxScaler()
features_scaled = scaler.fit_transform(df_clean[feature_cols].values)
target = df_clean[target_col].values.astype(np.float32)

print(f"Features shape: {features_scaled.shape}")
print(f"Target shape: {target.shape}, mean: {target.mean():.3f}")

## 2. CREATE SLIDING WINDOWS

In [ ]:
print("\n" + "=" * 60)
print("2. CREATING SLIDING WINDOWS")
print("=" * 60)

WINDOW_SIZE = 15  # Same as our paper (15 timesteps)
N_FEATURES = len(feature_cols)

def create_sequences(features, target, window_size):
    X, y = [], []
    for i in range(len(features) - window_size):
        X.append(features[i:i + window_size])
        y.append(target[i + window_size])
    return np.array(X), np.array(y)

X, y = create_sequences(features_scaled, target, WINDOW_SIZE)
print(f"Sequences: X={X.shape}, y={y.shape}")

# Chronological split (80/20)
split_idx = int(len(X) * 0.8)
X_train, X_test = X[:split_idx], X[split_idx:]
y_train, y_test = y[:split_idx], y[split_idx:]

# Validation set: last 10% of training
val_idx = int(len(X_train) * 0.9)
X_val = X_train[val_idx:]
y_val = y_train[val_idx:]
X_train_final = X_train[:val_idx]
y_train_final = y_train[:val_idx]

print(f"Train: {X_train_final.shape}, Val: {X_val.shape}, Test: {X_test.shape}")
print(f"Target distribution - Train: {y_train_final.mean():.3f}, Test: {y_test.mean():.3f}")

## 3. DEFINE MODELS

In [ ]:
print("\n" + "=" * 60)
print("3. DEFINING MODELS")
print("=" * 60)

import tensorflow as tf
from tensorflow.keras.models import Model
from tensorflow.keras.layers import (Input, LSTM, Dense, Dropout, Layer,
                                      Concatenate, Activation)
from tensorflow.keras.callbacks import EarlyStopping

# --- Attention Layer (same as our paper, compatible with Keras 3) ---
class AttentionLayer(Layer):
    def __init__(self, units=20, return_weights=False, **kwargs):
        super(AttentionLayer, self).__init__(**kwargs)
        self.units = units
        self.return_weights = return_weights

    def build(self, input_shape):
        self.W = self.add_weight(name='attention_weight',
                                  shape=(input_shape[-1], self.units),
                                  initializer='glorot_uniform', trainable=True)
        self.b = self.add_weight(name='attention_bias',
                                  shape=(self.units,),
                                  initializer='zeros', trainable=True)
        self.u = self.add_weight(name='attention_score',
                                  shape=(self.units, 1),
                                  initializer='glorot_uniform', trainable=True)
        super(AttentionLayer, self).build(input_shape)

    def call(self, x):
        # Bahdanau additive attention using tf ops directly
        score = tf.nn.tanh(tf.matmul(x, self.W) + self.b)
        attention_weights = tf.nn.softmax(tf.matmul(score, self.u), axis=1)
        context_vector = x * attention_weights
        context_vector = tf.reduce_sum(context_vector, axis=1)
        if self.return_weights:
            return context_vector, attention_weights
        return context_vector

    def get_config(self):
        config = super(AttentionLayer, self).get_config()
        config.update({'units': self.units, 'return_weights': self.return_weights})
        return config


def build_baseline_lstm(input_shape, seed):
    """Baseline LSTM (same architecture as our paper)"""
    tf.random.set_seed(seed)
    np.random.seed(seed)

    inp = Input(shape=input_shape)
    x = LSTM(20, activation='tanh',
             kernel_initializer=tf.keras.initializers.GlorotUniform(seed=seed),
             return_sequences=False)(inp)
    x = Dropout(0.2)(x)
    out = Dense(1, activation='sigmoid')(x)

    model = Model(inputs=inp, outputs=out)
    model.compile(optimizer=tf.keras.optimizers.Adam(learning_rate=0.001),
                  loss='binary_crossentropy',
                  metrics=['accuracy'])
    return model


def build_attention_lstm(input_shape, seed):
    """Attention-LSTM (same architecture as our paper)"""
    tf.random.set_seed(seed)
    np.random.seed(seed)

    inp = Input(shape=input_shape)
    lstm_out = LSTM(20, activation='tanh',
                    kernel_initializer=tf.keras.initializers.GlorotUniform(seed=seed),
                    return_sequences=True)(inp)
    lstm_out = Dropout(0.2)(lstm_out)
    context = AttentionLayer(units=20)(lstm_out)
    out = Dense(1, activation='sigmoid')(context)

    model = Model(inputs=inp, outputs=out)
    model.compile(optimizer=tf.keras.optimizers.Adam(learning_rate=0.001),
                  loss='binary_crossentropy',
                  metrics=['accuracy'])
    return model


def get_attention_weights(model, X):
    """Extract attention weights by rebuilding with return_weights=True"""
    # Find the attention layer and get its weights
    attn_layer = None
    for layer in model.layers:
        if isinstance(layer, AttentionLayer):
            attn_layer = layer
            break

    # Compute attention weights directly using tf ops
    # Get LSTM output (layer index 1 = LSTM, 2 = Dropout)
    # Build a function that runs input through LSTM+Dropout, then computes weights
    @tf.function
    def compute_weights(x):
        # Forward pass through LSTM
        lstm_out = model.layers[1](x)  # LSTM
        lstm_dropped = model.layers[2](lstm_out, training=False)  # Dropout
        # Compute attention weights
        score = tf.nn.tanh(tf.matmul(lstm_dropped, attn_layer.W) + attn_layer.b)
        weights = tf.nn.softmax(tf.matmul(score, attn_layer.u), axis=1)
        return weights

    # Process in batches to avoid OOM
    batch_size = 1000
    all_weights = []
    for i in range(0, len(X), batch_size):
        batch = X[i:i+batch_size]
        w = compute_weights(tf.constant(batch, dtype=tf.float32))
        all_weights.append(w.numpy())
    attn = np.concatenate(all_weights, axis=0)
    return attn.squeeze(-1)  # (n_samples, 15)

print("Models defined successfully.")
print(f"Input shape: ({WINDOW_SIZE}, {N_FEATURES})")

## 4. MULTI-SEED TRAINING (n=8)

In [ ]:
print("\n" + "=" * 60)
print("4. MULTI-SEED TRAINING (n=8)")
print("=" * 60)

SEEDS = [42, 123, 456, 789, 1010, 1213, 1415, 1617]
input_shape = (WINDOW_SIZE, N_FEATURES)

from sklearn.metrics import mean_absolute_error, r2_score, accuracy_score

baseline_results = []
attention_results = []
best_attention_model = None
best_attention_acc = -1

for i, seed in enumerate(SEEDS):
    print(f"\n--- Seed {seed} ({i+1}/8) ---")

    # Baseline LSTM
    model_b = build_baseline_lstm(input_shape, seed)
    es = EarlyStopping(monitor='val_loss', patience=10, min_delta=1e-5,
                       restore_best_weights=True, verbose=0)
    model_b.fit(X_train_final, y_train_final,
                validation_data=(X_val, y_val),
                epochs=100, batch_size=256,
                callbacks=[es], verbose=0)

    y_pred_b = model_b.predict(X_test, verbose=0).flatten()
    y_pred_class_b = (y_pred_b > 0.5).astype(int)
    mae_b = mean_absolute_error(y_test, y_pred_b)
    r2_b = r2_score(y_test, y_pred_b)
    acc_b = accuracy_score(y_test, y_pred_class_b)
    baseline_results.append({
        'seed': seed, 'MAE': mae_b, 'R2': r2_b, 'Accuracy': acc_b
    })
    print(f"  Baseline: MAE={mae_b:.4f}, R²={r2_b:.4f}, Acc={acc_b:.4f}")

    # Attention-LSTM
    model_a = build_attention_lstm(input_shape, seed)
    es = EarlyStopping(monitor='val_loss', patience=10, min_delta=1e-5,
                       restore_best_weights=True, verbose=0)
    model_a.fit(X_train_final, y_train_final,
                validation_data=(X_val, y_val),
                epochs=100, batch_size=256,
                callbacks=[es], verbose=0)

    y_pred_a = model_a.predict(X_test, verbose=0).flatten()
    y_pred_class_a = (y_pred_a > 0.5).astype(int)
    mae_a = mean_absolute_error(y_test, y_pred_a)
    r2_a = r2_score(y_test, y_pred_a)
    acc_a = accuracy_score(y_test, y_pred_class_a)
    attention_results.append({
        'seed': seed, 'MAE': mae_a, 'R2': r2_a, 'Accuracy': acc_a
    })
    print(f"  Attention: MAE={mae_a:.4f}, R²={r2_a:.4f}, Acc={acc_a:.4f}")

    # Keep best attention model
    if acc_a > best_attention_acc:
        best_attention_acc = acc_a
        best_attention_model = model_a
        best_attention_seed = seed

# Summary
df_baseline = pd.DataFrame(baseline_results)
df_attention = pd.DataFrame(attention_results)

print("\n" + "=" * 60)
print("MULTI-SEED RESULTS SUMMARY")
print("=" * 60)

# Convergence: R² > 0
baseline_converged = (df_baseline['R2'] > 0).sum()
attention_converged = (df_attention['R2'] > 0).sum()
print(f"\nBaseline convergence (R²>0): {baseline_converged}/8 ({baseline_converged/8*100:.1f}%)")
print(f"Attention convergence (R²>0): {attention_converged}/8 ({attention_converged/8*100:.1f}%)")

# More thresholds
for thresh in [0.0, 0.5, 0.7]:
    b_conv = (df_baseline['R2'] > thresh).sum()
    a_conv = (df_attention['R2'] > thresh).sum()
    print(f"  R²>{thresh}: Baseline {b_conv}/8, Attention {a_conv}/8")

print(f"\nBaseline stats:")
print(df_baseline.describe())
print(f"\nAttention stats:")
print(df_attention.describe())

print(f"\nBest attention model: seed {best_attention_seed}, Acc={best_attention_acc:.4f}")

# Save seed results
df_baseline.to_csv('baseline_seed_results_iraq.csv', index=False)
df_attention.to_csv('attention_seed_results_iraq.csv', index=False)

## 5. MC-DROPOUT UNCERTAINTY (both architectures)

In [ ]:
print("\n" + "=" * 60)
print("5. MC-DROPOUT UNCERTAINTY ESTIMATION")
print("=" * 60)

N_MC = 100

def mc_dropout_predict(model, X, n_mc=100):
    """MC-dropout: n_mc forward passes with dropout active"""
    predictions = []
    for _ in range(n_mc):
        pred = model(X, training=True)  # training=True keeps dropout active
        predictions.append(pred.numpy().flatten())
    predictions = np.array(predictions)  # (n_mc, n_samples)
    mean_pred = predictions.mean(axis=0)
    std_pred = predictions.std(axis=0)
    return mean_pred, std_pred

# MC-dropout on attention-LSTM
print("Running MC-dropout on attention-LSTM...")
attn_mc_mean, attn_mc_std = mc_dropout_predict(best_attention_model, X_test, N_MC)

# MC-dropout on best baseline
best_baseline_seed = df_baseline.loc[df_baseline['R2'].idxmax(), 'seed']
model_b_best = build_baseline_lstm(input_shape, int(best_baseline_seed))
es = EarlyStopping(monitor='val_loss', patience=10, min_delta=1e-5,
                   restore_best_weights=True, verbose=0)
model_b_best.fit(X_train_final, y_train_final,
                 validation_data=(X_val, y_val),
                 epochs=100, batch_size=256,
                 callbacks=[es], verbose=0)

print("Running MC-dropout on baseline LSTM...")
base_mc_mean, base_mc_std = mc_dropout_predict(model_b_best, X_test, N_MC)

# Uncertainty-error correlations
from scipy.stats import pearsonr

errors_attn = np.abs(y_test - attn_mc_mean)
errors_base = np.abs(y_test - base_mc_mean)

r_ue_attn, p_ue_attn = pearsonr(attn_mc_std, errors_attn)
r_ue_base, p_ue_base = pearsonr(base_mc_std, errors_base)

print(f"\nAttention MC-dropout: mean σ={attn_mc_std.mean():.6f}, UE correlation r={r_ue_attn:.4f} (p={p_ue_attn:.2e})")
print(f"Baseline MC-dropout:  mean σ={base_mc_std.mean():.6f}, UE correlation r={r_ue_base:.4f} (p={p_ue_base:.2e})")
print(f"Uncertainty reduction: {(1 - attn_mc_std.mean()/base_mc_std.mean())*100:.1f}%")

## 6. BOOTSTRAP SHAP ANALYSIS (200 instances)

In [ ]:
print("\n" + "=" * 60)
print("6. BOOTSTRAP SHAP ANALYSIS (200 instances)")
print("=" * 60)

import shap

# Stratified sampling: 200 test instances
from sklearn.model_selection import StratifiedShuffleSplit

# Create bins for stratification
temp_bins = pd.qcut(X_test[:, -1, 0], q=3, labels=False, duplicates='drop')  # temperature
humid_bins = pd.qcut(X_test[:, -1, 1], q=4, labels=False, duplicates='drop')  # humidity
target_bins = (y_test > 0.5).astype(int)  # pump on/off

# Combined stratification key
strat_key = temp_bins * 100 + humid_bins * 10 + target_bins

N_SHAP = 200
try:
    sss = StratifiedShuffleSplit(n_splits=1, test_size=N_SHAP, random_state=42)
    _, shap_idx = next(sss.split(X_test, strat_key))
except:
    # Fallback: random sample if stratification fails
    np.random.seed(42)
    shap_idx = np.random.choice(len(X_test), N_SHAP, replace=False)

X_shap = X_test[shap_idx]
y_shap = y_test[shap_idx]
print(f"SHAP sample: {X_shap.shape}, target mean: {y_shap.mean():.3f}")

# Background data (100 training samples, stratified)
np.random.seed(42)
bg_idx = np.random.choice(len(X_train_final), 100, replace=False)
X_background = X_train_final[bg_idx]

# Flatten for KernelExplainer
X_shap_flat = X_shap.reshape(N_SHAP, -1)
X_bg_flat = X_background.reshape(100, -1)

def model_predict_flat(X_flat):
    X_3d = X_flat.reshape(-1, WINDOW_SIZE, N_FEATURES)
    return best_attention_model.predict(X_3d, verbose=0).flatten()

print("Initializing KernelExplainer...")
explainer = shap.KernelExplainer(model_predict_flat, X_bg_flat)

print("Computing SHAP values for 200 instances...")
shap_values = explainer.shap_values(X_shap_flat, nsamples=1000)
shap_values_3d = np.array(shap_values).reshape(N_SHAP, WINDOW_SIZE, N_FEATURES)
print(f"SHAP values shape: {shap_values_3d.shape}")

# Feature importance (mean |SHAP| per feature)
feature_importance = np.mean(np.abs(shap_values_3d), axis=(0, 1))
print("\nFeature Importance (mean |SHAP|):")
for fname, imp in sorted(zip(feature_cols, feature_importance), key=lambda x: -x[1]):
    print(f"  {fname}: {imp:.6f}")

total_importance = feature_importance.sum()
print("\nFeature Importance (%):")
for fname, imp in sorted(zip(feature_cols, feature_importance), key=lambda x: -x[1]):
    print(f"  {fname}: {imp/total_importance*100:.1f}%")

# Temporal importance (mean |SHAP| per timestep, summed over features)
temporal_importance = np.mean(np.abs(shap_values_3d), axis=(0, 2))
temporal_pct = temporal_importance / temporal_importance.sum() * 100
print("\nTemporal Importance (%):")
for t in range(WINDOW_SIZE):
    print(f"  t-{WINDOW_SIZE-t}: {temporal_pct[t]:.1f}%")

last3_pct = temporal_pct[-3:].sum()
print(f"\nLast 3 timesteps: {last3_pct:.1f}%")

## 7. BOOTSTRAP RESAMPLING (1,000 iterations)

In [ ]:
print("\n" + "=" * 60)
print("7. BOOTSTRAP RESAMPLING (1,000 iterations)")
print("=" * 60)

N_BOOTSTRAP = 1000
np.random.seed(42)

boot_feature_importance = np.zeros((N_BOOTSTRAP, N_FEATURES))
boot_temporal_importance = np.zeros((N_BOOTSTRAP, WINDOW_SIZE))

for b in range(N_BOOTSTRAP):
    idx = np.random.choice(N_SHAP, N_SHAP, replace=True)
    boot_shap = shap_values_3d[idx]
    boot_feature_importance[b] = np.mean(np.abs(boot_shap), axis=(0, 1))
    boot_temporal_importance[b] = np.mean(np.abs(boot_shap), axis=(0, 2))

    if (b + 1) % 200 == 0:
        print(f"  Bootstrap iteration {b+1}/{N_BOOTSTRAP}")

# Feature importance with CIs
print("\nFeature Importance with 95% CIs:")
feature_ci = {}
for i, fname in enumerate(feature_cols):
    mean_imp = boot_feature_importance[:, i].mean()
    std_imp = boot_feature_importance[:, i].std()
    ci_lower = np.percentile(boot_feature_importance[:, i], 2.5)
    ci_upper = np.percentile(boot_feature_importance[:, i], 97.5)
    ci_width = ci_upper - ci_lower
    feature_ci[fname] = {
        'mean': mean_imp, 'std': std_imp,
        'ci_lower': ci_lower, 'ci_upper': ci_upper, 'ci_width': ci_width
    }
    print(f"  {fname}: {mean_imp:.6f} [{ci_lower:.6f}, {ci_upper:.6f}] (width={ci_width:.6f})")

# SE for top feature
top_feature = max(feature_ci.keys(), key=lambda k: feature_ci[k]['mean'])
top_se = feature_ci[top_feature]['std']
print(f"\nTop feature: {top_feature}, SE = {top_se:.6f}")

## 8. SHAP-ATTENTION CONVERGENCE

In [ ]:
print("\n" + "=" * 60)
print("8. SHAP-ATTENTION CONVERGENCE")
print("=" * 60)

# Extract attention weights for SHAP samples
attn_weights = get_attention_weights(best_attention_model, X_shap)
print(f"Attention weights shape: {attn_weights.shape}")

# Mean attention per timestep
mean_attention = attn_weights.mean(axis=0)
attn_pct = mean_attention / mean_attention.sum() * 100

# SHAP temporal profile (normalized)
shap_temporal_norm = temporal_importance / temporal_importance.sum()
attn_temporal_norm = mean_attention / mean_attention.sum()

# Pearson correlation
r_convergence, p_convergence = pearsonr(shap_temporal_norm, attn_temporal_norm)
print(f"\nSHAP-Attention correlation: r = {r_convergence:.4f}, p = {p_convergence:.2e}")

# Bootstrap CI for correlation
boot_r = []
for b in range(N_BOOTSTRAP):
    idx = np.random.choice(N_SHAP, N_SHAP, replace=True)
    boot_shap_temp = np.mean(np.abs(shap_values_3d[idx]), axis=(0, 2))
    boot_attn_temp = attn_weights[idx].mean(axis=0)
    boot_shap_norm = boot_shap_temp / boot_shap_temp.sum()
    boot_attn_norm = boot_attn_temp / boot_attn_temp.sum()
    r, _ = pearsonr(boot_shap_norm, boot_attn_norm)
    boot_r.append(r)

r_ci_lower = np.percentile(boot_r, 2.5)
r_ci_upper = np.percentile(boot_r, 97.5)
print(f"Bootstrap 95% CI for r: [{r_ci_lower:.4f}, {r_ci_upper:.4f}]")

# Per-sample divergence
print("\nPer-sample divergence analysis...")
divergences = []
for i in range(N_SHAP):
    shap_profile = np.abs(shap_values_3d[i]).mean(axis=1)  # (15,)
    shap_norm = shap_profile / (shap_profile.sum() + 1e-10)
    attn_norm = attn_weights[i] / (attn_weights[i].sum() + 1e-10)
    di = np.sum(np.abs(shap_norm - attn_norm))
    divergences.append(di)
divergences = np.array(divergences)

# Divergence-error correlation
y_pred_shap = best_attention_model.predict(X_shap, verbose=0).flatten()
errors_shap = np.abs(y_shap - y_pred_shap)

q75 = np.percentile(divergences, 75)
q25 = np.percentile(divergences, 25)
high_div = divergences > q75
low_div = divergences < q25

mae_high = errors_shap[high_div].mean()
mae_low = errors_shap[low_div].mean()
error_increase = (mae_high - mae_low) / mae_low * 100

print(f"Divergence threshold (75th pct): {q75:.4f}")
print(f"High-divergence MAE: {mae_high:.4f}")
print(f"Low-divergence MAE: {mae_low:.4f}")
print(f"Error increase: {error_increase:.1f}%")

r_div_err, p_div_err = pearsonr(divergences, errors_shap)
print(f"Divergence-error correlation: r={r_div_err:.4f}, p={p_div_err:.2e}")

## 9. GENERATE FIGURES

In [ ]:
print("\n" + "=" * 60)
print("9. GENERATING FIGURES")
print("=" * 60)

import matplotlib.pyplot as plt
import matplotlib
matplotlib.rcParams['font.size'] = 11

# --- Figure 1: Feature Importance with CIs ---
fig, ax = plt.subplots(figsize=(8, 5))
sorted_features = sorted(feature_ci.keys(), key=lambda k: feature_ci[k]['mean'], reverse=True)
means = [feature_ci[f]['mean'] for f in sorted_features]
ci_lowers = [feature_ci[f]['ci_lower'] for f in sorted_features]
ci_uppers = [feature_ci[f]['ci_upper'] for f in sorted_features]
yerr_low = [m - cl for m, cl in zip(means, ci_lowers)]
yerr_high = [cu - m for m, cu in zip(means, ci_uppers)]

bars = ax.barh(range(len(sorted_features)), means, xerr=[yerr_low, yerr_high],
               capsize=5, color='steelblue', edgecolor='black', alpha=0.8)
ax.set_yticks(range(len(sorted_features)))
ax.set_yticklabels(sorted_features)
ax.set_xlabel('Mean |SHAP| Value')
ax.set_title('Feature Importance with 95% Bootstrap CIs\n(IoT Agriculture 2024 - Iraq)')
ax.invert_yaxis()
plt.tight_layout()
plt.savefig('iraq_shap_feature_importance.png', dpi=150, bbox_inches='tight')
plt.show()
print("Saved: iraq_shap_feature_importance.png")

# --- Figure 2: Temporal Importance Heatmap ---
temporal_by_feature = np.mean(np.abs(shap_values_3d), axis=0)  # (15, 6)
fig, ax = plt.subplots(figsize=(10, 5))
im = ax.imshow(temporal_by_feature.T, aspect='auto', cmap='YlOrRd')
ax.set_xticks(range(WINDOW_SIZE))
ax.set_xticklabels([f't-{WINDOW_SIZE-t}' for t in range(WINDOW_SIZE)])
ax.set_yticks(range(N_FEATURES))
ax.set_yticklabels(feature_cols)
ax.set_xlabel('Timestep')
ax.set_ylabel('Feature')
ax.set_title('SHAP Temporal Importance Heatmap\n(IoT Agriculture 2024 - Iraq)')
plt.colorbar(im, label='Mean |SHAP|')
plt.tight_layout()
plt.savefig('iraq_temporal_importance_heatmap.png', dpi=150, bbox_inches='tight')
plt.show()
print("Saved: iraq_temporal_importance_heatmap.png")

# --- Figure 3: SHAP vs Attention Temporal Profiles ---
fig, ax = plt.subplots(figsize=(8, 5))
timesteps = [f't-{WINDOW_SIZE-t}' for t in range(WINDOW_SIZE)]
ax.plot(range(WINDOW_SIZE), shap_temporal_norm * 100, 'bo-', label='SHAP', linewidth=2)
ax.plot(range(WINDOW_SIZE), attn_temporal_norm * 100, 'rs--', label='Attention', linewidth=2)
ax.set_xticks(range(WINDOW_SIZE))
ax.set_xticklabels(timesteps, rotation=45)
ax.set_ylabel('Temporal Weight (%)')
ax.set_xlabel('Timestep')
ax.set_title(f'SHAP vs Attention Temporal Profiles (r={r_convergence:.2f})\n(IoT Agriculture 2024 - Iraq)')
ax.legend()
ax.grid(True, alpha=0.3)
plt.tight_layout()
plt.savefig('iraq_shap_attention_convergence.png', dpi=150, bbox_inches='tight')
plt.show()
print("Saved: iraq_shap_attention_convergence.png")

# --- Figure 4: Multi-seed comparison ---
fig, axes = plt.subplots(1, 2, figsize=(12, 5))

# R² comparison
ax = axes[0]
x = range(len(SEEDS))
ax.bar([i - 0.15 for i in x], df_baseline['R2'], width=0.3, label='Baseline', color='salmon')
ax.bar([i + 0.15 for i in x], df_attention['R2'], width=0.3, label='Attention', color='steelblue')
ax.set_xticks(x)
ax.set_xticklabels(SEEDS, rotation=45)
ax.set_ylabel('R²')
ax.set_xlabel('Random Seed')
ax.set_title('R² Across Seeds')
ax.legend()
ax.axhline(y=0, color='black', linestyle='--', alpha=0.5)
ax.grid(True, alpha=0.3)

# Accuracy comparison
ax = axes[1]
ax.bar([i - 0.15 for i in x], df_baseline['Accuracy'], width=0.3, label='Baseline', color='salmon')
ax.bar([i + 0.15 for i in x], df_attention['Accuracy'], width=0.3, label='Attention', color='steelblue')
ax.set_xticks(x)
ax.set_xticklabels(SEEDS, rotation=45)
ax.set_ylabel('Accuracy')
ax.set_xlabel('Random Seed')
ax.set_title('Accuracy Across Seeds')
ax.legend()
ax.grid(True, alpha=0.3)

plt.suptitle('Multi-Seed Training Stability (IoT Agriculture 2024 - Iraq)', fontsize=13)
plt.tight_layout()
plt.savefig('iraq_multi_seed_comparison.png', dpi=150, bbox_inches='tight')
plt.show()
print("Saved: iraq_multi_seed_comparison.png")

## 10. SAVE ALL RESULTS

In [ ]:
print("\n" + "=" * 60)
print("10. SAVING ALL RESULTS")
print("=" * 60)

results = {
    'dataset': 'IoT Agriculture 2024 (Tikrit University, Iraq)',
    'climate': 'Arid',
    'target': 'Watering pump ON/OFF',
    'n_samples': int(len(df_clean)),
    'n_sequences': int(len(X)),
    'n_train': int(len(X_train_final)),
    'n_test': int(len(X_test)),
    'window_size': WINDOW_SIZE,
    'n_features': N_FEATURES,
    'feature_names': feature_cols,
    'n_seeds': len(SEEDS),

    # Multi-seed results
    'baseline_convergence_r2_gt_0': int(baseline_converged),
    'attention_convergence_r2_gt_0': int(attention_converged),
    'baseline_mean_r2': float(df_baseline['R2'].mean()),
    'baseline_std_r2': float(df_baseline['R2'].std()),
    'attention_mean_r2': float(df_attention['R2'].mean()),
    'attention_std_r2': float(df_attention['R2'].std()),
    'baseline_mean_accuracy': float(df_baseline['Accuracy'].mean()),
    'attention_mean_accuracy': float(df_attention['Accuracy'].mean()),

    # MC-dropout
    'mc_dropout_attention_mean_std': float(attn_mc_std.mean()),
    'mc_dropout_baseline_mean_std': float(base_mc_std.mean()),
    'mc_dropout_uncertainty_reduction_pct': float((1 - attn_mc_std.mean()/base_mc_std.mean())*100),
    'mc_dropout_ue_r_attention': float(r_ue_attn),
    'mc_dropout_ue_r_baseline': float(r_ue_base),

    # SHAP feature importance
    'feature_importance': {fname: float(imp) for fname, imp in zip(feature_cols, feature_importance)},
    'feature_importance_pct': {fname: float(imp/total_importance*100) for fname, imp in zip(feature_cols, feature_importance)},
    'top_feature': top_feature,
    'top_feature_se': float(top_se),
    'feature_ci': {fname: {k: float(v) for k, v in ci.items()} for fname, ci in feature_ci.items()},

    # SHAP temporal profile
    'shap_temporal_profile': [float(x) for x in shap_temporal_norm],
    'attn_temporal_profile': [float(x) for x in attn_temporal_norm],
    'last3_timestep_pct': float(last3_pct),

    # Convergence
    'pearson_r': float(r_convergence),
    'pearson_p': float(p_convergence),
    'r_ci_lower': float(r_ci_lower),
    'r_ci_upper': float(r_ci_upper),
    'n_shap_samples': N_SHAP,
    'n_bootstrap': N_BOOTSTRAP,

    # Divergence
    'divergence_threshold_75pct': float(q75),
    'mae_high_divergence': float(mae_high),
    'mae_low_divergence': float(mae_low),
    'divergence_error_increase_pct': float(error_increase),
    'divergence_error_r': float(r_div_err),
    'divergence_error_p': float(p_div_err),
}

with open('iraq_cross_validation_results.json', 'w') as f:
    json.dump(results, f, indent=2)
print("Saved: iraq_cross_validation_results.json")

# Save feature importance CSV
fi_df = pd.DataFrame({
    'Feature': feature_cols,
    'Importance': feature_importance,
    'Importance_pct': feature_importance / total_importance * 100,
    'CI_Lower': [feature_ci[f]['ci_lower'] for f in feature_cols],
    'CI_Upper': [feature_ci[f]['ci_upper'] for f in feature_cols],
    'CI_Width': [feature_ci[f]['ci_width'] for f in feature_cols],
})
fi_df = fi_df.sort_values('Importance', ascending=False)
fi_df.to_csv('iraq_shap_feature_importance.csv', index=False)
print("Saved: iraq_shap_feature_importance.csv")

# Save temporal importance CSV
temp_df = pd.DataFrame(
    np.mean(np.abs(shap_values_3d), axis=0),
    columns=feature_cols,
    index=[f't-{WINDOW_SIZE-t}' for t in range(WINDOW_SIZE)]
)
temp_df.to_csv('iraq_shap_temporal_importance.csv')
print("Saved: iraq_shap_temporal_importance.csv")

## FINAL SUMMARY

In [ ]:
print("\n" + "=" * 60)
print("CROSS-DATASET VALIDATION SUMMARY")
print("=" * 60)
print(f"Dataset: IoT Agriculture 2024 (Iraq, arid climate)")
print(f"Task: Watering pump prediction (binary)")
print(f"Sequences: {len(X)} ({len(X_train_final)} train, {len(X_test)} test)")
print(f"")
print(f"MULTI-SEED STABILITY:")
print(f"  Baseline convergence: {baseline_converged}/8 ({baseline_converged/8*100:.1f}%)")
print(f"  Attention convergence: {attention_converged}/8 ({attention_converged/8*100:.1f}%)")
print(f"  Baseline mean R²: {df_baseline['R2'].mean():.4f} ± {df_baseline['R2'].std():.4f}")
print(f"  Attention mean R²: {df_attention['R2'].mean():.4f} ± {df_attention['R2'].std():.4f}")
print(f"")
print(f"SHAP ANALYSIS (200 instances):")
print(f"  Top feature: {top_feature} ({feature_importance.max()/total_importance*100:.1f}%)")
print(f"  Top feature SE: {top_se:.6f}")
print(f"  Last-3 timesteps: {last3_pct:.1f}%")
print(f"")
print(f"SHAP-ATTENTION CONVERGENCE:")
print(f"  r = {r_convergence:.4f} [{r_ci_lower:.4f}, {r_ci_upper:.4f}], p = {p_convergence:.2e}")
print(f"")
print(f"DIVERGENCE MONITORING:")
print(f"  High-div MAE: {mae_high:.4f} vs Low-div: {mae_low:.4f} (+{error_increase:.1f}%)")
print(f"  Divergence-error r = {r_div_err:.4f}")
print(f"")
print(f"MC-DROPOUT:")
print(f"  Attention σ: {attn_mc_std.mean():.6f}, Baseline σ: {base_mc_std.mean():.6f}")
print(f"  Uncertainty reduction: {(1 - attn_mc_std.mean()/base_mc_std.mean())*100:.1f}%")
print(f"")
print("DONE! Download iraq_cross_validation_results.json and all figures.")